# SageMaker vs Vertex AI — Managed ML Pipelines

## Mental Model

**SageMaker** and **Vertex AI** are the managed ML control planes of AWS and GCP.

They solve the same business problem:

- train models on managed infrastructure
- orchestrate repeatable ML pipelines
- track artifacts and versions
- deploy models for inference
- monitor production behavior

For a Data Engineer, the real value is not just “training a model.”  
It is building the **reliable path from raw data → features → training job → validation → registration → deployment**.

This notebook uses a Citi-style telemetry domain:

- **endpoints**: 10,000 rows
- **metrics**: 500,000 rows
- **alerts**: 25,000 rows
- **Narrative**: 6,000+ API endpoints monitored for latency, error rate, throughput; alerts escalate through severity tiers

We will:

1. prepare a small anomaly-detection training dataset from PostgreSQL
2. define a SageMaker training job
3. define a SageMaker pipeline JSON
4. define and optionally trigger a Vertex AI CustomJob
5. compare the platforms in an engineering-friendly way

This notebook is written to be **safe and executable top-to-bottom**.  
Where cloud permissions, buckets, roles, or services are unavailable, cells will **report status cleanly instead of crashing**.

In [ ]:
import json
import os
import time
import uuid
from pathlib import Path
from textwrap import dedent

import boto3
import pandas as pd
import psycopg2
from botocore.exceptions import BotoCoreError, ClientError
from google.api_core.exceptions import GoogleAPICallError
import google.auth
from google.cloud import aiplatform
from sklearn.ensemble import IsolationForest

AWS_PROFILE = "study"
AWS_REGION = "us-east-1"
AWS_ACCOUNT = "357811130281"

GCP_PROJECT = "citi-de-learning"
GCP_CREDENTIALS = r"D:/Workspace/Technologies/_setup/gcp_key.json"
GCP_REGION = "us-central1"

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "de_telemetry",
    "user": "de_admin",
    "password": "DeAdmin2026!",
}

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = GCP_CREDENTIALS

print("AWS profile:", AWS_PROFILE)
print("AWS region:", AWS_REGION)
print("AWS account:", AWS_ACCOUNT)
print("GCP project:", GCP_PROJECT)
print("GCP credentials:", os.environ["GOOGLE_APPLICATION_CREDENTIALS"])

In [ ]:
def get_pg_conn():
    return psycopg2.connect(**DB_CONFIG)

def query_df(sql: str, params=None) -> pd.DataFrame:
    with get_pg_conn() as conn:
        return pd.read_sql(sql, conn, params=params)

dataset_sql = '''
WITH metrics_daily AS (
    SELECT
        m.endpoint_id,
        DATE_TRUNC('day', m.timestamp) AS event_day,
        AVG(CASE WHEN m.metric_name = 'latency' THEN m.value END) AS avg_latency,
        AVG(CASE WHEN m.metric_name = 'error_rate' THEN m.value END) AS avg_error_rate,
        AVG(CASE WHEN m.metric_name = 'throughput' THEN m.value END) AS avg_throughput
    FROM metrics m
    GROUP BY m.endpoint_id, DATE_TRUNC('day', m.timestamp)
),
alerts_daily AS (
    SELECT
        a.endpoint_id,
        DATE_TRUNC('day', a.created_at) AS event_day,
        COUNT(*)::int AS alert_count,
        MAX(
            CASE LOWER(a.severity)
                WHEN 'critical' THEN 4
                WHEN 'sev1' THEN 4
                WHEN 'high' THEN 3
                WHEN 'sev2' THEN 3
                WHEN 'medium' THEN 2
                WHEN 'low' THEN 1
                ELSE 0
            END
        ) AS severity_rank
    FROM alerts a
    GROUP BY a.endpoint_id, DATE_TRUNC('day', a.created_at)
)
SELECT
    md.endpoint_id,
    md.event_day,
    COALESCE(md.avg_latency, 0.0) AS avg_latency,
    COALESCE(md.avg_error_rate, 0.0) AS avg_error_rate,
    COALESCE(md.avg_throughput, 0.0) AS avg_throughput,
    COALESCE(ad.alert_count, 0) AS alert_count,
    COALESCE(ad.severity_rank, 0) AS severity_rank,
    CASE WHEN COALESCE(ad.alert_count, 0) > 0 THEN 1 ELSE 0 END AS label_alerted
FROM metrics_daily md
LEFT JOIN alerts_daily ad
  ON ad.endpoint_id = md.endpoint_id
 AND ad.event_day = md.event_day
ORDER BY md.event_day DESC, md.endpoint_id
LIMIT 5000
'''

dataset_df = query_df(dataset_sql)
display(dataset_df.head())
print("Training dataset shape:", dataset_df.shape)

In [ ]:
feature_cols = ["avg_latency", "avg_error_rate", "avg_throughput", "alert_count", "severity_rank"]
X = dataset_df[feature_cols].fillna(0.0)
model = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1,
)
model.fit(X)
pred = model.predict(X)
dataset_df["is_anomaly"] = (pred == -1).astype(int)

print("Feature columns:", feature_cols)
print("Sample anomaly count:", int(dataset_df["is_anomaly"].sum()))
display(dataset_df.head())

In [ ]:
# AWS clients
aws_session_status = "not-initialized"
s3_client = None
sm_client = None
sts_client = None
aws_identity = None

try:
    boto3.setup_default_session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
    sts_client = boto3.client("sts")
    s3_client = boto3.client("s3")
    sm_client = boto3.client("sagemaker")
    aws_identity = sts_client.get_caller_identity()
    aws_session_status = "ok"
except Exception as e:
    aws_session_status = f"unavailable: {type(e).__name__}: {e}"

print("AWS session status:", aws_session_status)
if aws_identity:
    print("AWS caller identity:", aws_identity)

In [ ]:
# Prepare SageMaker job definition.
job_suffix = uuid.uuid4().hex[:8]
sagemaker_training_job_name = f"citi-telemetry-iforest-{job_suffix}"
sagemaker_role_arn = f"arn:aws:iam::{AWS_ACCOUNT}:role/service-role/AmazonSageMaker-ExecutionRole"
sagemaker_bucket = f"citi-de-learning-{AWS_ACCOUNT}-us-east-1"
s3_input_uri = f"s3://{sagemaker_bucket}/ml/telemetry/input/"
s3_output_uri = f"s3://{sagemaker_bucket}/ml/telemetry/output/"
sagemaker_image_uri = "683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"

training_job_request = {
    "TrainingJobName": sagemaker_training_job_name,
    "AlgorithmSpecification": {
        "TrainingImage": sagemaker_image_uri,
        "TrainingInputMode": "File",
    },
    "RoleArn": sagemaker_role_arn,
    "InputDataConfig": [
        {
            "ChannelName": "train",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": s3_input_uri,
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "ContentType": "text/csv",
            "CompressionType": "None",
            "RecordWrapperType": "None",
        }
    ],
    "OutputDataConfig": {"S3OutputPath": s3_output_uri},
    "ResourceConfig": {
        "InstanceType": "ml.m5.large",
        "InstanceCount": 1,
        "VolumeSizeInGB": 20,
    },
    "StoppingCondition": {"MaxRuntimeInSeconds": 1800},
    "HyperParameters": {
        "sagemaker_program": "train.py",
        "sagemaker_submit_directory": s3_input_uri,
    },
    "Environment": {
        "PROJECT_NAME": "citi-telemetry-anomaly",
        "MODEL_TYPE": "IsolationForest",
    },
}

print("SageMaker training job request:")
print(json.dumps(training_job_request, indent=2))

In [ ]:
# Optionally create SageMaker training job.
# This cell is execution-safe: if bucket/role/code is not available, it reports cleanly.

sagemaker_training_status = "not-started"

if sm_client is None:
    sagemaker_training_status = "skipped: AWS client unavailable"
else:
    try:
        # Best effort submit. In many local study setups this may fail due to missing bucket/role/code package.
        sm_client.create_training_job(**training_job_request)
        poll_attempts = 0
        max_attempts = 20
        while poll_attempts < max_attempts:
            desc = sm_client.describe_training_job(TrainingJobName=sagemaker_training_job_name)
            sagemaker_training_status = desc.get("TrainingJobStatus", "Unknown")
            if sagemaker_training_status in {"Completed", "Failed", "Stopped"}:
                break
            poll_attempts += 1
            time.sleep(3)
    except Exception as e:
        sagemaker_training_status = f"submission-not-completed: {type(e).__name__}: {e}"

print(f"SageMaker training job: {sagemaker_training_status}")

In [ ]:
# SageMaker pipeline definition (JSON) with Processing -> Training -> Evaluation -> Model Registration
pipeline_name = f"citi-telemetry-pipeline-{job_suffix}"

pipeline_definition = {
    "Version": "2020-12-01",
    "Metadata": {},
    "Parameters": [
        {
            "Name": "ProcessingInstanceType",
            "Type": "String",
            "DefaultValue": "ml.m5.large",
        },
        {
            "Name": "TrainingInstanceType",
            "Type": "String",
            "DefaultValue": "ml.m5.large",
        },
        {
            "Name": "ModelApprovalStatus",
            "Type": "String",
            "DefaultValue": "PendingManualApproval",
        },
    ],
    "PipelineExperimentConfig": {
        "ExperimentName": {"Get": "Execution.PipelineName"},
        "TrialName": {"Get": "Execution.PipelineExecutionId"},
    },
    "Steps": [
        {
            "Name": "Processing",
            "Type": "Processing",
            "Arguments": {
                "ProcessingResources": {
                    "ClusterConfig": {
                        "InstanceCount": 1,
                        "InstanceType": {"Get": "Parameters.ProcessingInstanceType"},
                        "VolumeSizeInGB": 20,
                    }
                },
                "AppSpecification": {
                    "ImageUri": sagemaker_image_uri,
                    "ContainerEntrypoint": ["python3", "preprocess.py"],
                },
                "RoleArn": sagemaker_role_arn,
            },
        },
        {
            "Name": "Training",
            "Type": "Training",
            "Arguments": training_job_request,
        },
        {
            "Name": "Evaluation",
            "Type": "Processing",
            "Arguments": {
                "ProcessingResources": {
                    "ClusterConfig": {
                        "InstanceCount": 1,
                        "InstanceType": "ml.m5.large",
                        "VolumeSizeInGB": 20,
                    }
                },
                "AppSpecification": {
                    "ImageUri": sagemaker_image_uri,
                    "ContainerEntrypoint": ["python3", "evaluate.py"],
                },
                "RoleArn": sagemaker_role_arn,
            },
        },
        {
            "Name": "ModelRegistration",
            "Type": "RegisterModel",
            "Arguments": {
                "ModelPackageGroupName": "citi-telemetry-anomaly-detector",
                "ModelApprovalStatus": {"Get": "Parameters.ModelApprovalStatus"},
                "InferenceSpecification": {
                    "Containers": [
                        {
                            "Image": sagemaker_image_uri,
                            "ModelDataUrl": s3_output_uri,
                        }
                    ],
                    "SupportedContentTypes": ["text/csv"],
                    "SupportedResponseMIMETypes": ["text/csv"],
                },
            },
        },
    ],
}

print("SageMaker Pipeline Definition JSON:")
print(json.dumps(pipeline_definition, indent=2)[:6000])

In [ ]:
# Create and optionally start the SageMaker pipeline.
sagemaker_pipeline_status = "not-created"

if sm_client is None:
    sagemaker_pipeline_status = "skipped: AWS client unavailable"
else:
    try:
        sm_client.create_pipeline(
            PipelineName=pipeline_name,
            PipelineDefinition=json.dumps(pipeline_definition),
            RoleArn=sagemaker_role_arn,
        )
        execution = sm_client.start_pipeline_execution(PipelineName=pipeline_name)
        sagemaker_pipeline_status = f"started: {execution.get('PipelineExecutionArn', 'unknown-arn')}"
    except Exception as e:
        sagemaker_pipeline_status = f"not-started: {type(e).__name__}: {e}"

print("SageMaker step types:")
print("- Processing: feature prep / validation / evaluation")
print("- Training: managed model training")
print("- Evaluation: metrics and gate checks")
print("- RegisterModel: publish approved artifact to registry")
print("SageMaker pipeline status:", sagemaker_pipeline_status)

In [ ]:
# Initialize Vertex AI SDK
vertex_init_status = "not-initialized"
vertex_auth_status = "unknown"

try:
    creds, detected_project = google.auth.default()
    vertex_auth_status = f"ok (default project={detected_project})"
except Exception as e:
    vertex_auth_status = f"unavailable: {type(e).__name__}: {e}"

try:
    aiplatform.init(project=GCP_PROJECT, location=GCP_REGION)
    vertex_init_status = "ok"
except Exception as e:
    vertex_init_status = f"unavailable: {type(e).__name__}: {e}"

print("Vertex auth status:", vertex_auth_status)
print("Vertex init status:", vertex_init_status)

In [ ]:
# Define a Vertex AI CustomJob.
vertex_job_display_name = f"citi-vertex-iforest-{job_suffix}"
vertex_staging_bucket = f"gs://{GCP_PROJECT}-vertex-staging"

worker_pool_specs = [
    {
        "machine_spec": {
            "machine_type": "n1-standard-4",
        },
        "replica_count": 1,
        "container_spec": {
            "image_uri": "us-docker.pkg.dev/vertex-ai/training/scikit-learn-cpu.1-0:latest",
            "command": ["python", "-c"],
            "args": [
                dedent(
                    '''
                    import json
                    print(json.dumps({
                        "job": "vertex-ai-customjob",
                        "model": "IsolationForest",
                        "domain": "citi-telemetry",
                        "message": "CustomJob container started successfully."
                    }))
                    '''
                ).strip()
            ],
        },
    }
]

print("Vertex worker pool specs:")
print(json.dumps(worker_pool_specs, indent=2))

In [ ]:
# Optionally submit Vertex AI CustomJob.
vertex_job_status = "not-started"

try:
    custom_job = aiplatform.CustomJob(
        display_name=vertex_job_display_name,
        worker_pool_specs=worker_pool_specs,
        staging_bucket=vertex_staging_bucket,
    )
    try:
        # sync=False so notebook remains responsive and safe in local study mode
        custom_job.run(sync=False)
        state = getattr(custom_job, "state", None)
        vertex_job_status = str(state or "submitted")
    except Exception as inner_e:
        vertex_job_status = f"submission-not-completed: {type(inner_e).__name__}: {inner_e}"
except Exception as outer_e:
    vertex_job_status = f"definition-not-created: {type(outer_e).__name__}: {outer_e}"

print(f"Vertex AI job: {vertex_job_status}")

In [ ]:
comparison_df = pd.DataFrame([
    {
        "dimension": "managed notebooks",
        "SageMaker": "Studio / notebook instances",
        "Vertex AI": "Workbench / Colab Enterprise",
        "Citi recommendation": "Use whichever matches cloud landing zone"
    },
    {
        "dimension": "pipeline orchestration",
        "SageMaker": "Strong native pipelines",
        "Vertex AI": "Strong native pipelines + KFP alignment",
        "Citi recommendation": "Vertex if KFP portability matters; SageMaker if AWS-native"
    },
    {
        "dimension": "feature store",
        "SageMaker": "Native feature store",
        "Vertex AI": "Native feature store",
        "Citi recommendation": "Both work; choose by cloud standard"
    },
    {
        "dimension": "model registry",
        "SageMaker": "Model registry and package groups",
        "Vertex AI": "Model registry in Vertex AI",
        "Citi recommendation": "Both are enterprise-capable"
    },
    {
        "dimension": "monitoring",
        "SageMaker": "Model monitor + CloudWatch ecosystem",
        "Vertex AI": "Model monitoring + Cloud Logging ecosystem",
        "Citi recommendation": "Integrate with existing observability stack"
    },
    {
        "dimension": "cost model",
        "SageMaker": "Usage-based, can sprawl without guardrails",
        "Vertex AI": "Usage-based, similar risk",
        "Citi recommendation": "Enforce quotas, tags, budgets, teardown automation"
    },
    {
        "dimension": "Citi recommendation",
        "SageMaker": "Best when data + deployment are AWS-centered",
        "Vertex AI": "Best when analytics / AI platform is GCP-centered",
        "Citi recommendation": "Prefer the platform aligned to enterprise cloud strategy"
    },
])

display(comparison_df)

## What Just Happened

**SageMaker** is AWS’s end-to-end ML platform.  
**Vertex AI** is GCP’s equivalent.

Both provide:

- managed training
- pipeline orchestration
- model registry
- serving and monitoring

The practical Data Engineering angle is this:

- Data Science owns the model choice and experimentation
- Data Engineering owns the **data path**, **feature path**, **deployment automation**, and **operational reliability**

In a Citi-style environment, the real production asset is not just the model artifact.  
It is the **repeatable governed pipeline** that can retrain, validate, register, and deploy safely.